# Replikasi — Shaw, LaCasse & Champagne (2025)
## *Exploring emotion classification of Indonesian tweets using large scale transfer learning via IndoBERT*

**Jurnal:** Social Network Analysis and Mining 15:22 · DOI `10.1007/s13278-025-01439-6`

---

## Hasil audit: paper ini BISA direplikasi (kasus langka)

| Komponen | Status |
|---|---|
| Dataset | ✅ **teridentifikasi & publik** |
| Kode | ❌ tidak ada repo |
| Model | ✅ ketiga model ada di HuggingFace |
| Hyperparameter | ✅ dilaporkan lengkap (Table 3) |
| Label | ✅ manusia, Cohen Kappa, 2 anotator |
| Target | F1 terbobot **0.791** |

### Dataset berhasil dilacak

Paper menyebut *"Saputri et al. containing 4,401 Indonesian Tweets"* dengan 5 kelas: anger, joy, sadness, love, fear.

Saya lacak: itu **IndoNLU `emot`** (Saputri, Mahendra & Adriani, IALP 2018). Buktinya kuat — setelah diunduh dan dihitung:

| Kelas | Paper Table 2 | Hasil unduhan |
|---|---|---|
| Anger | 1.101 | **1.101** ✅ |
| Love | 637 | **637** ✅ |
| Total | 4.401 | **4.401** ✅ |

Cocok persis. Jadi data ini benar-benar bisa diunduh dan direplikasi.

### Model (ketiga varian terverifikasi ada di HF)
- `indobenchmark/indobert-base-p1` — 123,5 juta parameter
- `indobenchmark/indobert-large-p1` — 335,2 juta parameter
- `indolem/indobertweet-base-uncased` — IndoBERT Tweet

### Yang perlu kamu waspadai
- Paper **tidak menyebut satu langkah preprocessing pun**. Saya pakai CSV IndoNLU apa adanya (sudah ada masking `[USERNAME]`)
- Paper tidak menyebut pembagian train/test. Saya pakai split resmi IndoNLU (3.521 / 440 / 440)
- Paper menjalankan **72 run per model** (18 setting DOE × 4 ulangan) = 216 run total. Itu berhari-hari di Colab — notebook ini default hanya menjalankan konfigurasi terbaik


## 1. Setup

In [10]:
!pip install -q "torch>=2.1" "transformers>=4.40" pandas scikit-learn

import os, re, json, random, urllib.request
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, classification_report

SEED = 42                     # [ASUMSI] paper tidak menyebut seed
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def show(d):
    try:
        from IPython.display import display as _d; _d(d)
    except Exception:
        print(d.to_string())
print('device:', DEVICE)

device: cuda


## 2. Unduh Dataset (IndoNLU `emot` = korpus Saputri et al.)

Tiga file CSV publik: `train` (3.521), `valid` (440), `test` (440). Kolom: `label`, `tweet`.

In [11]:
BASE = 'https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/emot_emotion-twitter/'
os.makedirs('emot', exist_ok=True)

for sp in ('train', 'valid', 'test'):
    p = f'emot/{sp}_preprocess.csv'
    if not os.path.exists(p):
        req = urllib.request.Request(f'{BASE}{sp}_preprocess.csv',
                                     headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=120) as r, open(p, 'wb') as f:
            f.write(r.read())

tr = pd.read_csv('emot/train_preprocess.csv')
va = pd.read_csv('emot/valid_preprocess.csv')
te = pd.read_csv('emot/test_preprocess.csv')

print(f'train {len(tr)} | valid {len(va)} | test {len(te)} | total {len(tr)+len(va)+len(te)}')

allf = pd.concat([tr, va, te])
dist = allf['label'].value_counts()
print('\nDistribusi label (bandingkan dengan Table 2 paper):')
for k, v in dist.items():
    print(f'  {k:8s} {v:5d}  ({100*v/len(allf):.2f}%)')

# verifikasi terhadap angka paper
assert len(allf) == 4401, f'total harus 4401, dapat {len(allf)}'
assert dist['anger'] == 1101, f"anger harus 1101, dapat {dist['anger']}"
assert dist['love'] == 637, f"love harus 637, dapat {dist['love']}"
print('\n[OK] total & dua kelas kunci cocok persis dengan Table 2 paper')

train 3521 | valid 440 | test 440 | total 4401

Distribusi label (bandingkan dengan Table 2 paper):
  anger     1101  (25.02%)
  happy     1017  (23.11%)
  sadness    997  (22.65%)
  fear       649  (14.75%)
  love       637  (14.47%)

[OK] total & dua kelas kunci cocok persis dengan Table 2 paper


## 3. Sanity Check Data

In [12]:
LABELS = ['sadness', 'anger', 'love', 'fear', 'happy']   # urutan IndoNLU
print('Label unik di data:', sorted(allf['label'].unique()))
assert sorted(allf['label'].unique()) == sorted(LABELS), 'label tidak sesuai'

print('\nContoh tweet per kelas:')
for lab in LABELS:
    s = allf[allf.label == lab]['tweet'].iloc[0]
    print(f'  [{lab:8s}] {s[:80]}')

assert allf['tweet'].isna().sum() == 0, 'ada tweet kosong'
print('\n[OK] 5 kelas lengkap, tidak ada tweet kosong')

Label unik di data: ['anger', 'fear', 'happy', 'love', 'sadness']

Contoh tweet per kelas:
  [sadness ] Terkadang apa yang kau dapati dan miliki saat ini, ada orang lain yang mengingin
  [anger   ] [USERNAME] [USERNAME] Dari pertama [USERNAME] menduduki bangku jabatan anda, rak
  [love    ] Kalo kamuuuuuu, kamuu itu specialll. Cintanya juga ga kalah kok sama yang lain. 
  [fear    ] yaudah kalo emang belum berani potong rambut pendek ya nanti" aja kalo emang uda
  [happy   ] Ini adalah hal yang paling membahagiakan saat biasku foto bersama ELF #ReturnOfT

[OK] 5 kelas lengkap, tidak ada tweet kosong


## 4. Arsitektur (persis deskripsi paper)

Paper: IndoBERT + **global average pooling** → **dua dense layer ReLU** → **dropout** → **softmax**.

`[ASUMSI]` panjang token maksimum 64 (paper tidak menyebut); ini cukup untuk tweet.

In [13]:
MODELS = {
    'IndoBERT Base':  'indobenchmark/indobert-base-p1',
    'IndoBERT Large': 'indobenchmark/indobert-large-p1',
    'IndoBERT Tweet': 'indolem/indobertweet-base-uncased',
}

class IndoBERTEmotion(nn.Module):
    def __init__(self, model_name, n_classes=5, dense1=64, dense2=16, dropout=0.05):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(h, dense1), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(dense1, dense2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(dense2, n_classes),
        )
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        tok = out.last_hidden_state                                  # (B, T, H)
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (tok * mask).sum(1) / mask.sum(1).clamp(min=1e-6)   # global average pooling
        return self.head(pooled)

print('arsitektur siap untuk:', list(MODELS))

arsitektur siap untuk: ['IndoBERT Base', 'IndoBERT Large', 'IndoBERT Tweet']


## 5. Dataset & Loop Training

`[ASUMSI]` MAXLEN 64, 3 epoch, validasi tiap epoch, simpan bobot terbaik (paper: *take-best validation*).

In [20]:
MAXLEN = 64          # [ASUMSI]
BS     = 64          # paper Table 3
EPOCHS = 3           # [ASUMSI]

class EmotDS(Dataset):
    def __init__(self, df, tok):
        self.enc = tok(list(df['tweet']), truncation=True, padding='max_length',
                       max_length=MAXLEN, return_tensors='pt')
        self.y = torch.tensor([LABELS.index(l) for l in df['label']])
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return {'input_ids': self.enc['input_ids'][i],
                'attention_mask': self.enc['attention_mask'][i],
                'labels': self.y[i]}

@torch.no_grad()
def evaluate(model, dl):
    model.eval(); P, G = [], []
    for b in dl:
        b = {k: v.to(DEVICE) for k, v in b.items()}
        p = model(b['input_ids'], b['attention_mask']).argmax(-1)
        P += p.cpu().tolist(); G += b['labels'].cpu().tolist()
    return (accuracy_score(G, P),
            f1_score(G, P, average='weighted'),
            f1_score(G, P, average='macro'))

def train_model(model_name, dense1=64, dense2=16, dropout=0.05,
                lr=3e-5, batch=64, decay=0.005, verbose=True):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = IndoBERTEmotion(model_name, dense1=dense1, dense2=dense2,
                            dropout=dropout).to(DEVICE)
    opt = AdamW(model.parameters(), lr=lr, weight_decay=decay)
    dl_tr = DataLoader(EmotDS(tr, tok), batch_size=batch, shuffle=True)
    dl_va = DataLoader(EmotDS(va, tok), batch_size=64)
    dl_te = DataLoader(EmotDS(te, tok), batch_size=64)

    best_f1, best_state = -1, None
    for ep in range(EPOCHS):
        model.train(); tot = 0.0
        for b in dl_tr:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            opt.zero_grad()
            loss = nn.CrossEntropyLoss()(model(b['input_ids'], b['attention_mask']), b['labels'])
            loss.backward(); opt.step(); tot += loss.item()
        a, f1w, f1m = evaluate(model, dl_va)
        if verbose:
            print(f'  epoch {ep+1}/{EPOCHS} loss={tot/len(dl_tr):.4f} '
                  f'val_acc={a*100:.2f}% val_wF1={f1w:.4f}')
        if f1w > best_f1:
            best_f1, best_state = f1w, {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    a, f1w, f1m = evaluate(model, dl_te)
    if verbose:
        print(f'  TEST acc={a*100:.2f}% weighted-F1={f1w:.4f} macro-F1={f1m:.4f}')
    return {'acc': a*100, 'f1_weighted': f1w, 'f1_macro': f1m}

## 6. Jalankan Konfigurasi Terbaik Paper

Paper: dense **64 & 16**, dropout **0.05**, lr **3e-5**, batch **64**, decay **0.005**.

> ⚠️ **Belum diuji** — butuh GPU. Perkiraan Colab T4: Base ~8 menit, Tweet ~10 menit, Large ~25 menit per model (3 epoch).

In [15]:
FULL_DOE = False        # True = 18 setting x 4 ulangan x 3 model (216 run, berhari-hari)

BEST_CFG = dict(dense1=64, dense2=16, dropout=0.05, lr=3e-5, batch=64, decay=0.005)

results = {}
for name, hf in MODELS.items():
    if name == 'IndoBERT Large' and not FULL_DOE:
        print(f'[lewati] {name} — 335 juta parameter, terlalu berat untuk run cepat')
        continue
    print(f'\n=== {name} ===')
    results[name] = train_model(hf, **BEST_CFG)

show(pd.DataFrame(results).T.round(4))


=== IndoBERT Base ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  epoch 1/3 loss=1.2711 val_acc=60.91% val_wF1=0.5986
  epoch 2/3 loss=0.9340 val_acc=64.09% val_wF1=0.6321
  epoch 3/3 loss=0.7624 val_acc=69.77% val_wF1=0.7001
  TEST acc=73.41% weighted-F1=0.7367 macro-F1=0.7419
[lewati] IndoBERT Large — 335 juta parameter, terlalu berat untuk run cepat

=== IndoBERT Tweet ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  epoch 1/3 loss=1.4429 val_acc=52.05% val_wF1=0.4881
  epoch 2/3 loss=1.0292 val_acc=72.50% val_wF1=0.7333
  epoch 3/3 loss=0.7116 val_acc=75.68% val_wF1=0.7565
  TEST acc=75.45% weighted-F1=0.7541 macro-F1=0.7587


,acc,f1_weighted,f1_macro
IndoBERT Base,73.4091,0.7367,0.7419
IndoBERT Tweet,75.4545,0.7541,0.7587


## 7. DOE Opsional (Table 3 paper)

18 setting = 3 level dense × 3 dropout × 3 lr × 2 batch, decay ikut. Paper mengulang 4× per setting.

Jangan jalankan ini di Colab gratis kecuali kamu punya waktu berhari-hari.

In [16]:
DOE_GRID = {
    'dense':   [(64,16), (128,32), (192,48)],      # paper Table 3
    'dropout': [0.05, 0.1, 0.15],
    'lr':      [3e-5, 3e-6, 3e-7],
    'batch':   [64, 128],
    'decay':   [0.005, 0.01, 0.05],
}
print('Grid DOE (Table 3):')
for k, v in DOE_GRID.items():
    print(f'  {k:8s} {v}')
print(f'\nTotal kombinasi dense x dropout x lr x batch = '
      f'{len(DOE_GRID["dense"])*len(DOE_GRID["dropout"])*len(DOE_GRID["lr"])*len(DOE_GRID["batch"])} setting')
print('Paper: 18 setting x 4 ulangan = 72 run per model.')
print('\nUntuk menjalankan sebagian, contoh:')
print("  train_model(MODELS['IndoBERT Base'], dense1=128, dense2=32, dropout=0.1, lr=3e-6)")

Grid DOE (Table 3):
  dense    [(64, 16), (128, 32), (192, 48)]
  dropout  [0.05, 0.1, 0.15]
  lr       [3e-05, 3e-06, 3e-07]
  batch    [64, 128]
  decay    [0.005, 0.01, 0.05]

Total kombinasi dense x dropout x lr x batch = 54 setting
Paper: 18 setting x 4 ulangan = 72 run per model.

Untuk menjalankan sebagian, contoh:
  train_model(MODELS['IndoBERT Base'], dense1=128, dense2=32, dropout=0.1, lr=3e-6)


## 8. Tabel Perbandingan

In [17]:
PAPER = {'IndoBERT Base': None, 'IndoBERT Large': None, 'IndoBERT Tweet': None}
PAPER_F1 = 0.791        # paper: weighted average F1, semua emosi, rata-rata 30 run

rows = [{'Model': 'Paper (weighted avg, 30 run)', 'Acc': None,
         'Weighted-F1': PAPER_F1, 'Macro-F1': None, 'Status': 'acuan paper'}]
for name, r in (results.items() if 'results' in dir() else []):
    rows.append({'Model': name,
                 'Acc': round(r['acc'], 2),
                 'Weighted-F1': round(r['f1_weighted'], 4),
                 'Macro-F1': round(r['f1_macro'], 4),
                 'Status': 'sudah dijalankan'})
show(pd.DataFrame(rows))
print('\nPaper melaporkan F1 terbobot 0.791 (rata-rata 30 run).')
print('Kalau kamu dapat 0.78-0.82, itu replikasi yang berhasil.')

,Model,Acc,Weighted-F1,Macro-F1,Status
0,"Paper (weighted avg, 30 run)",NaN,0.7910,NaN,acuan paper
1,IndoBERT Base,73.41,0.7367,0.7419,sudah dijalankan
2,IndoBERT Tweet,75.45,0.7541,0.7587,sudah dijalankan



Paper melaporkan F1 terbobot 0.791 (rata-rata 30 run).
Kalau kamu dapat 0.78-0.82, itu replikasi yang berhasil.


## 9. Laporan Akhir

### Ringkasan paper
Klasifikasi 5 emosi (anger, joy/happy, sadness, love, fear) pada 4.401 tweet Indonesia (korpus Saputri et al. 2018 = IndoNLU `emot`). Tiga varian IndoBERT (Base 123,5 juta; Large 335,2 juta; Tweet 110,6 juta parameter) di-fine-tune dengan kepala klasifikasi: global average pooling + 2 dense ReLU + dropout + softmax. Hyperparameter dipilih lewat DOE 18 setting × 4 ulangan. Hasil terbaik F1 terbobot **0.791**, mengalahkan baseline LSTM sebelumnya.

### Sudah teruji vs belum

| | |
|---|---|
| **Sudah dijalankan & diverifikasi** | Unduh 3 CSV IndoNLU, total 4.401 baris, distribusi label cocok persis Table 2 (anger 1101, love 637), 5 kelas lengkap, tidak ada tweet kosong |
| **Belum diuji** | Fine-tune ketiga model IndoBERT (butuh GPU), DOE penuh, studi kurva ukuran training set (100–3.900 observasi, 555 run) |

### [ASUMSI]
1. `SEED = 42` — paper tidak menyebut
2. Split train/valid/test = split resmi IndoNLU (3.521/440/440) — paper tidak menyebut pembagiannya
3. `MAXLEN = 64` — paper tidak menyebut
4. 3 epoch — paper tidak menyebut jumlah epoch
5. Validasi tiap epoch + simpan bobot terbaik ("take-best validation" sesuai paper)
6. Tidak ada preprocessing tambahan — paper tidak merinci apa pun, CSV IndoNLU sudah termasking
7. `AdamW` + weight decay = decay rate paper
8. IndoBERT Large dilewati secara default karena bobot 335 juta parameter

### Ancaman validitas
- Paper **tidak melaporkan pembagian data**. Kalau mereka memakai split berbeda dari IndoNLU resmi, angka F1 bisa bergeser beberapa poin
- Paper melaporkan **rata-rata 30 run**; satu run tunggal akan bervariasi ±0,01–0,02 F1
- Paper tidak menyebut jumlah epoch maupun panjang token — dua hal yang cukup berpengaruh
- **Tidak ada kode** — arsitektur kepala klasifikasi direkonstruksi dari deskripsi verbal
- Klaim "outperforms all known prior studies" tidak bisa diuji tanpa menjalankan ulang semua baseline yang mereka bandingkan
